# Section 16.4 Attention Mechanisms

In [1]:
import numpy as np
import tensorflow as tf

class PositionalEncoding(tf.keras.layers.Layer):
    def __init__(self, steps_max: int, dims_max: int, dtype=tf.float32, **kwargs):
        super().__init__(dtype=dtype, **kwargs)
        if dims_max % 2 == 1:
            dims_max += 1
        word_positions, dim = np.meshgrid(np.arange(steps_max), np.arange(dims_max // 2))
        pos_emb = np.empty((1, steps_max, dims_max))
        pos_emb[0, :, ::2] = np.sin(word_positions / 10_000 ** (2 * dim / dims_max)).T
        pos_emb[0, :, 1::2] = np.cos(word_positions / 10_000 ** (2 * dim / dims_max)).T
        self.positional_embedding = tf.constant(pos_emb.astype(self.dtype))

    def __call__(self, inputs):
        shape = tf.shape(inputs)
        return inputs + self.positional_embedding[:, :shape[-2], :shape[-1]]

2023-04-11 09:30:41.829116: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2 AVX AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [ ]:
size_embed = 512
steps_max = 500
size_vocab = 10_000

encoder_inputs = tf.keras.layers.Input(shape=[None], dtype=np.int32)
decoder_inputs = tf.keras.layers.Input(shape=[None], dtype=np.int32)

embed_layer = tf.keras.layers.Embedding(input_dim=size_vocab, output_dim=size_embed)
encoder_embeddings = embed_layer(encoder_inputs)
decoder_embeddings = embed_layer(decoder_inputs)

positional_encoding_layer = PositionalEncoding(steps_max=steps_max, dims_max=size_embed)
encoder_in = positional_encoding_layer(encoder_embeddings)
decoder_in = positional_encoding_layer(decoder_embeddings)

In [ ]:
n_blocks = 6
z = encoder_in
for N in range(n_blocks):
    z = tf.keras.layers.Attention(use_scale=True)([z, z])

encoder_outputs = z
z = decoder_in
for N in range(n_blocks):
    z = tf.keras.layers.Attention(use_scale=True, causal=True)([z, z])
    z = tf.keras.layers.Attention(use_scale=True)([z, encoder_outputs])

outputs = tf.keras.layers.TimeDistributed(tf.keras.layers.Dense(size_vocab, activation="softmax"))(z)